In [19]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any

import torch
import numpy as np
import datasets
import fasttext
import fasttext.util
from transformers import BertTokenizer, BertModel

In [20]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [21]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [4]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]

def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab(test_corpus)

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

In [5]:
def one_hot_vectorization(text: str, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[int]:
    words =  text.split()
    vector = [0] * len(vocab)

    for word in words:
        if word in vocab_index:
            idx = vocab_index[word]
            vector[idx] = 1

    return vector

def test_one_hot_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for word in words_in_text:
            if word in vocab_index:
                idx = vocab_index[word]
                if result[idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [6]:
assert test_one_hot_vectorization(test_corpus, vocab, vocab_index)

One-Hot-Vectors test PASSED


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [7]:
def bag_of_words_vectorization(text: str) -> Dict[str, int]:
    words = text.split()
    return dict(Counter(words))
    
def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [8]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [9]:
def tf_idf_vectorization(text: str, corpus: List[str] = None, vocab: List[str] = None, vocab_index: Dict[str, int] = None) -> List[float]:
    words = text.split()
    tf_counts = Counter(words)
    total_terms = len(words)
    tf = {word: tf_counts[word] / total_terms for word in tf_counts}

    N = len(corpus)
    idf = {}
    for word in vocab:
        # количество документов, где встречается слово
        df = sum(1 for doc in corpus if word in doc.split())
        idf[word] = math.log((N / (df + 1)))

    vector = [0.0] * len(vocab)
    for word in tf:
        if word in vocab_index:
            idx = vocab_index[word]
            vector[idx] = tf[word] * idf[word]

    return vector

def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [10]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED


In [11]:
def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:
    cooc = defaultdict(Counter)
    word_counts = Counter()

    for doc in corpus:
        tokens =  doc.split()
        for i, word in enumerate(tokens):
            word_counts[word] += 1
            # контекстное окно
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            context_words = tokens[start:i] + tokens[i+1:end]
            for ctx in context_words:
                cooc[word][ctx] += 1
                
    total_cooc = sum(sum(c.values()) for c in cooc.values())
    total_words = sum(word_counts.values())

    ppmi_matrix = defaultdict(dict)
    for w in cooc:
        for c in cooc[w]:
            p_wc = cooc[w][c] / total_cooc
            p_w = word_counts[w] / total_words
            p_c = word_counts[c] / total_words
            pmi = math.log((p_wc / (p_w * p_c)) + 1e-12)
            ppmi_matrix[w][c] = max(0.0, pmi)

    tokens = normalize_pretokenize_text(text)
    vector = [0.0] * len(vocab)

    for word in tokens:
        if word in ppmi_matrix:
            for ctx, val in ppmi_matrix[word].items():
                if ctx in vocab_index:
                    vector[vocab_index[ctx]] += val

    if len(tokens) > 0:
        vector = [v / len(tokens) for v in vector]

    return vector

def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [12]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED


In [27]:
def get_fasttext_embeddings(text: str, model_path: str = None, model: any = None) -> List[np.ndarray]:
    model = fasttext.load(model_path)

    embeddings = []
    for word in text: # tokens:
        if word in model.wv:
            embeddings.append(model.wv[word])
        else:
            # если слово вне словаря, то усредняем по символам
            char_vectors = [model.wv[c] for c in word if c in model.wv]
            if char_vectors:
                embeddings.append(np.mean(char_vectors, axis=0))
            else:
                embeddings.append(np.zeros(model.vector_size))
    return embeddings

In [14]:
def get_bert_embeddings(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:

    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)

    model.eval()
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        outputs = model(**inputs)
        last_hidden_state = outputs.last_hidden_state  # [batch, seq_len, hidden_size]
        if pool_method == 'cls':
            # первый токен CLS
            embeddings = last_hidden_state[:, 0, :].squeeze(0)
        elif pool_method == 'mean':
            # усреднение по всем токенам
            embeddings = last_hidden_state.mean(dim=1).squeeze(0)
        else:
            raise ValueError("pool_method должен быть 'cls' или 'mean'")

    return embeddings.cpu().numpy()

In [15]:
def vectorize_dataset(
    dataset_name: str = "imdb",
    vectorizer_type: str = "bow",
    split: str = "train",
    sample_size: int = 2500
) -> Tuple[Any, List, List]:

    dataset = datasets.load_dataset(dataset_name, split=split)

    if sample_size:
        dataset = dataset.select(range(min(sample_size, len(dataset))))

    texts = [item['text'] for item in dataset if 'text' in item and item['text'].strip()]
    labels = [item['label'] for item in dataset if 'label' in item]

    def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
        all_words = []
        for text in texts:
            words = normalize_pretokenize_text(text)
            all_words.extend(words)
        vocab = sorted(set(all_words))
        vocab_index = {word: idx for idx, word in enumerate(vocab)}
        return vocab, vocab_index

    vocab, vocab_index = build_vocab(texts)

    vectorized_data = []
    for text in texts:
        if vectorizer_type == "one_hot":
            vectorized_data.append(one_hot_vectorization(text, vocab, vocab_index))
        elif vectorizer_type == "bow":
            bow_dict = bag_of_words_vectorization(text)
            vector = [bow_dict.get(word, 0) for word in vocab]
            vectorized_data.append(vector)
        elif vectorizer_type == "tfidf":
            vectorized_data.append(tf_idf_vectorization(text, texts, vocab, vocab_index))
        elif vectorizer_type == "ppmi":
            vectorized_data.append(ppmi_vectorization(text, texts, vocab, vocab_index))
        elif vectorizer_type == "fasttext":
            embeddings = get_fasttext_embeddings(text)
            if embeddings:
                avg_embedding = np.mean(embeddings, axis=0)
                vectorized_data.append(avg_embedding.tolist())
            else:
                vectorized_data.append([0] * 300)
        elif vectorizer_type == "bert":
            embedding = get_bert_embeddings(text)
            vectorized_data.append(embedding.tolist())
        else:
            raise ValueError(f"Unknown vectorizer type: {vectorizer_type}")

    ##### Мое исправление
    return vocab, vectorized_data, labels
    #####

In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
import numpy as np
# import warnings
# warnings.filterwarnings("ignore")

def train(
    embeddings_method="bow",
    test_size=0.2,
    val_size=0.2,
    cv_folds=5
):
    vocab, X, y = vectorize_dataset("imdb", embeddings_method, "train")
    _, X_test, y_test = vectorize_dataset("imdb", embeddings_method, "test")

    X = np.array(X, dtype=np.float32)
    X_test = np.array(X_test, dtype=np.float32)
    y = np.array(y)
    y_test = np.array(y_test)


    if embeddings_method in ["tfidf", "ppmi", "fasttext", "bert"]:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        X_test = scaler.transform(X_test)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=val_size, random_state=42, stratify=y
    )

    model = CatBoostClassifier(
        iterations=50,
        depth=3,
        learning_rate=0.2,
        loss_function='Logloss',
        eval_metric='F1',
        verbose=50,
        random_seed=42,
        task_type='GPU'
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val))

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(classification_report(y_test, y_pred, digits=3))

In [ ]:
for embeddings_method in ["bow", "one_hot", "tfidf", "ppmi", "fasttext", "bert"]:
    train(embeddings_method=embeddings_method)